# Beta `m8_xgb` Threshold And Variant Search

This diagnostic/improvement notebook is deliberately isolated from the main
journal workflow. It does not edit Notebook 2, `experiment_config.yaml`, or
production helpers.

Inputs:
- final Dataset Alpha and Dataset Beta;
- current journal experiment config;
- existing journal helper functions for data loading and m8 training.

Outputs are written only under:
`notebooks/99_Misc/outputs/02_beta_m8_xgb_threshold_and_variant_search/`.

The notebook first diagnoses XGB1 day probabilities and XGB2 interval
probabilities, then tries split-Beta threshold calibration, per-site threshold
calibration, site-normalised feature variants, and a small capped
hyperparameter set. Rows marked `all_beta_upper_bound` are optimistic and are
not publication-ready validation.


In [ ]:
from __future__ import annotations

import copy
import json
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score, precision_recall_curve

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "publication" / "2_journal_article").exists():
    ROOT = ROOT.parent
ARTICLE_ROOT = ROOT / "publication" / "2_journal_article"
NOTEBOOK_ROOT = ARTICLE_ROOT / "notebooks"
MISC_ROOT = NOTEBOOK_ROOT / "99_Misc"
OUTPUT_ROOT = MISC_ROOT / "outputs" / "02_beta_m8_xgb_threshold_and_variant_search"
CSV_DIR = OUTPUT_ROOT / "csv"
FIG_DIR = OUTPUT_ROOT / "figures"
HTML_DIR = OUTPUT_ROOT / "html_examples"
for folder in [CSV_DIR, FIG_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
import _experiment_helpers as h

JCOL = getattr(
    h,
    "JOURNAL_COLORS",
    {
        "orange": "#eb932c",
        "dark_blue": "#22303d",
        "grey": "#2F4D67",
        "light_grey": "#5C7D99",
        "light_white": "#ebe3e3",
    },
)
plt.rcParams.update(
    {
        "font.family": "Arial",
        "axes.edgecolor": JCOL["dark_blue"],
        "axes.labelcolor": JCOL["dark_blue"],
        "xtick.color": JCOL["dark_blue"],
        "ytick.color": JCOL["dark_blue"],
    }
)

cfg = h.load_config(ARTICLE_ROOT)
alpha = h.load_dataset(ARTICLE_ROOT, cfg, "alpha")
beta = h.load_dataset(ARTICLE_ROOT, cfg, "beta")

baseline_t1 = float(cfg["correction"]["m8_xgb"]["xgb1_day"]["threshold"])
baseline_t2 = float(cfg["correction"]["m8_xgb"]["xgb2_timestamp"]["threshold"])

BETA_FOLDS = [
    {
        "fold_id": "tune_2023Q4_2024Q1_eval_2024Q2Q3",
        "tune_start": "2023-10-01",
        "tune_end": "2024-03-31",
        "eval_start": "2024-04-01",
        "eval_end": "2024-09-30",
    },
    {
        "fold_id": "tune_2024Q2Q3_eval_2023Q4_2024Q1",
        "tune_start": "2024-04-01",
        "tune_end": "2024-09-30",
        "eval_start": "2023-10-01",
        "eval_end": "2024-03-31",
    },
]

print(f"Repository root: {ROOT}")
print(f"Article root:    {ARTICLE_ROOT}")
print(f"Output root:     {OUTPUT_ROOT}")
print(f"Alpha rows: {len(alpha):,} | Beta rows: {len(beta):,}")
print(f"Current thresholds: XGB1={baseline_t1:.6f}, XGB2={baseline_t2:.6f}")
beta_rpf_days = beta.groupby(["substation_id", "date"])["label_interval"].any().groupby("substation_id").sum()
display(beta_rpf_days.sort_values(ascending=False).to_frame("rpf_days"))


## Controls

The defaults run a bounded but useful search. Disable the variant flags below
if you only want the fastest probability and threshold diagnostics.


In [ ]:
RUN_SITE_NORMALISED_VARIANTS = True
RUN_LIGHT_HYPERPARAMETER_VARIANTS = True
WRITE_PLOTLY_HTML_EXAMPLES = True

XGB1_THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 19), 3)
XGB2_THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 19), 3)
SITE_MIN_POSITIVE_DAYS_FOR_SITE_THRESHOLD = 5

SITE_NORMALISATION_VARIANTS = [
    {
        "variant_id": "site_joint_p95",
        "description": "Scale net load and solar by each site's joint 95th percentile magnitude.",
        "scale_mode": "site_joint_p95",
    },
    {
        "variant_id": "site_daytime_joint_p95",
        "description": "Scale net load and solar by each site's daytime joint 95th percentile magnitude.",
        "scale_mode": "site_daytime_joint_p95",
    },
]

LIGHT_HYPERPARAMETER_VARIANTS = [
    {
        "variant_id": "hp_shallow_pos8",
        "description": "Shallower trees with stronger positive weighting.",
        "updates": {"eta": 0.08, "n_estimators": 350, "max_depth": 4, "scale_pos_weight": 8},
    },
    {
        "variant_id": "hp_deep_pos3",
        "description": "Deeper trees with lower positive weighting and lower learning rate.",
        "updates": {"eta": 0.05, "n_estimators": 350, "max_depth": 8, "scale_pos_weight": 3},
    },
]


## Helper Functions

These helpers expose probabilities that the production Notebook 2 output does
not store. They score XGB2 for all Beta days when requested, then apply
arbitrary XGB1/XGB2 thresholds without retraining.


In [ ]:
def write_csv(df: pd.DataFrame, name: str) -> Path:
    path = CSV_DIR / name
    df.to_csv(path, index=False)
    return path


def date_mask(df: pd.DataFrame, start: str, end: str) -> pd.Series:
    dates = pd.to_datetime(df["date"])
    return dates.between(pd.Timestamp(start), pd.Timestamp(end), inclusive="both")


def safe_div(num: float, den: float) -> float:
    return float(num / den) if den else 0.0


def binary_metrics(y_true: Any, y_pred: Any) -> dict[str, Any]:
    true = np.asarray(y_true, dtype=bool)
    pred = np.asarray(y_pred, dtype=bool)
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return {
        "support": int(len(true)),
        "positive_support": int(true.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def daytime_mask(df: pd.DataFrame) -> pd.Series:
    start = int(cfg["correction"]["interval_daytime_start_hour"])
    end = int(cfg["correction"]["interval_daytime_end_hour"])
    return df["hour"].between(start, end, inclusive="both")


def make_cfg_with_hyperparams(base_cfg: dict[str, Any], updates: dict[str, Any] | None = None) -> dict[str, Any]:
    work = copy.deepcopy(base_cfg)
    if updates:
        for section in ["xgb1_day", "xgb2_timestamp"]:
            work["correction"]["m8_xgb"][section].update(updates)
    return work


def site_scale_table(df: pd.DataFrame, mode: str) -> pd.Series:
    work = df.copy()
    if mode == "site_daytime_joint_p95":
        work = work.loc[daytime_mask(work)].copy()
    if mode not in {"site_joint_p95", "site_daytime_joint_p95"}:
        raise ValueError(f"Unknown scale mode: {mode}")
    values = work.assign(
        joint_magnitude=lambda x: np.maximum(
            pd.to_numeric(x["net_load_MW"], errors="coerce").abs(),
            pd.to_numeric(x["solar_MW"], errors="coerce").abs(),
        )
    )
    scale = values.groupby("substation_id")["joint_magnitude"].quantile(0.95)
    scale = scale.replace(0, np.nan).fillna(scale[scale > 0].median()).fillna(1.0)
    return scale


def transform_variant(df: pd.DataFrame, scale_mode: str | None) -> pd.DataFrame:
    if not scale_mode or scale_mode == "none":
        return df.copy()
    out = df.copy()
    scale = site_scale_table(out, scale_mode)
    factors = out["substation_id"].map(scale).astype(float).replace(0, 1.0)
    out["net_load_MW"] = out["net_load_MW"] / factors
    out["solar_MW"] = out["solar_MW"] / factors
    return out


def score_m8_probabilities(
    feature_df: pd.DataFrame,
    result_df: pd.DataFrame,
    bundle: dict[str, Any],
    run_cfg: dict[str, Any],
    score_xgb2_all_days: bool = True,
) -> pd.DataFrame:
    h._ensure_package_import(ARTICLE_ROOT)
    from pynrpf.plugins.m8_xgb import _align_features, _bundle_section, _feature_cfg
    from pynrpf.plugins.m8_xgb import build_xgb1_features, build_xgb2_features

    cols = run_cfg["columns"]
    m8_cfg = run_cfg["correction"]["m8_xgb"]
    work = h._model_input_columns(feature_df).copy()
    work["timestamp"] = h._parse_wall_clock(work["timestamp"])
    work["_pynrpf_gt_dummy"] = 1.0

    feature_cfg = _feature_cfg(work, cols["timestamp"], m8_cfg)
    clf1, feat_cols1, model_t1 = _bundle_section(bundle, "xgb1_day", m8_cfg["xgb1_day"]["threshold"])
    clf2, feat_cols2, _ = _bundle_section(bundle, "xgb2_timestamp", m8_cfg["xgb2_timestamp"]["threshold"])

    day_df, _, _ = build_xgb1_features(
        work,
        feature_cfg,
        cols["site"],
        cols["timestamp"],
        cols["net_load"],
        cols["solar"],
        "_pynrpf_gt_dummy",
    )
    prob_day = clf1.predict_proba(_align_features(day_df, feat_cols1).to_numpy(np.float32))[:, 1]
    # Keep the full XGB1 feature frame for XGB2. Dropping it here would force
    # the aligner to fill many training features with zero and invalidate the
    # diagnostic scorer.
    day_features_df = day_df.copy()
    day_features_df["prob_day"] = prob_day
    day_df = day_features_df[[cols["site"], "date", "prob_day"]].copy()
    candidate_keys = day_df[[cols["site"], "date"]].copy()
    if not score_xgb2_all_days:
        candidate_keys = day_df.loc[day_df["prob_day"] >= model_t1, [cols["site"], "date"]].copy()

    if candidate_keys.empty:
        ts_results = pd.DataFrame(columns=[cols["site"], cols["timestamp"], "prob_interval"])
    else:
        ts_df, _, _ = build_xgb2_features(
            work,
            feature_cfg,
            day_features_df,
            candidate_keys,
            cols["site"],
            cols["timestamp"],
            cols["net_load"],
            cols["solar"],
            "_pynrpf_gt_dummy",
        )
        if ts_df.empty:
            ts_results = pd.DataFrame(columns=[cols["site"], cols["timestamp"], "prob_interval"])
        else:
            prob_ts = clf2.predict_proba(_align_features(ts_df, feat_cols2).to_numpy(np.float32))[:, 1]
            ts_results = ts_df[[cols["site"], cols["timestamp"]]].copy()
            ts_results["prob_interval"] = prob_ts

    result = result_df.copy().reset_index(drop=True)
    result["_date_key"] = pd.to_datetime(result["date"]).dt.date
    day_map = day_df.set_index([cols["site"], "date"])["prob_day"]
    day_idx = pd.MultiIndex.from_frame(result[[cols["site"]]].assign(date=result["_date_key"]))
    result["prob_day"] = day_idx.map(day_map).astype(float)

    result_ts = h._parse_wall_clock(result[cols["timestamp"]])
    result_key = pd.DataFrame({cols["site"]: result[cols["site"]], cols["timestamp"]: result_ts})
    if ts_results.empty:
        result["prob_interval"] = np.nan
    else:
        merged = result_key.merge(ts_results, on=[cols["site"], cols["timestamp"]], how="left")
        result["prob_interval"] = pd.to_numeric(merged["prob_interval"], errors="coerce").to_numpy()
    return result.drop(columns=["_date_key"])


def apply_global_thresholds(scored: pd.DataFrame, t1: float, t2: float) -> pd.DataFrame:
    out = scored.copy()
    day_ok = out["prob_day"] >= float(t1)
    interval_ok = out["prob_interval"].fillna(-np.inf) >= float(t2)
    out["pred_day_candidate"] = day_ok.astype(bool)
    out["pred_interval"] = (day_ok & interval_ok).astype(bool)
    out["corrected_net_load_MW"] = np.where(out["pred_interval"], -out["net_load_MW"], out["net_load_MW"])
    return out


def apply_site_thresholds(scored: pd.DataFrame, site_thresholds: pd.DataFrame) -> pd.DataFrame:
    out = scored.copy()
    lookup = site_thresholds.set_index("substation_id")[["xgb1_threshold", "xgb2_threshold"]]
    out["_t1"] = out["substation_id"].map(lookup["xgb1_threshold"]).astype(float)
    out["_t2"] = out["substation_id"].map(lookup["xgb2_threshold"]).astype(float)
    day_ok = out["prob_day"] >= out["_t1"]
    interval_ok = out["prob_interval"].fillna(-np.inf) >= out["_t2"]
    out["pred_day_candidate"] = day_ok.astype(bool)
    out["pred_interval"] = (day_ok & interval_ok).astype(bool)
    out["corrected_net_load_MW"] = np.where(out["pred_interval"], -out["net_load_MW"], out["net_load_MW"])
    return out.drop(columns=["_t1", "_t2"])


def evaluate_predicted_frame(pred: pd.DataFrame, dataset: str, fold_id: str, result_type: str, variant_id: str, calibration: str) -> pd.DataFrame:
    day_df = pred.groupby(["substation_id", "date"], as_index=False).agg(
        label_day=("label_interval", "any"),
        pred_day=("pred_interval", "any"),
    )
    rows = []
    for level, values in [
        ("day", binary_metrics(day_df["label_day"], day_df["pred_day"])),
        ("interval", binary_metrics(pred.loc[daytime_mask(pred), "label_interval"], pred.loc[daytime_mask(pred), "pred_interval"])),
    ]:
        rows.append(
            {
                "dataset": dataset,
                "fold_id": fold_id,
                "result_type": result_type,
                "variant_id": variant_id,
                "calibration": calibration,
                "level": level,
                **values,
            }
        )
    return pd.DataFrame(rows)


## Threshold Search Helpers

The global threshold search ranks by final day-level F1. Per-site thresholds
fall back to the global thresholds where a site has too few positive or
negative days in the tuning half.


In [ ]:
def xgb1_threshold_sweep(scored: pd.DataFrame, fold_id: str, variant_id: str) -> pd.DataFrame:
    day_df = scored.groupby(["substation_id", "date"], as_index=False).agg(
        label_day=("label_interval", "any"),
        prob_day=("prob_day", "first"),
    )
    rows = []
    for t1 in XGB1_THRESHOLD_GRID:
        rows.append(
            {
                "fold_id": fold_id,
                "variant_id": variant_id,
                "xgb1_threshold": float(t1),
                **binary_metrics(day_df["label_day"], day_df["prob_day"] >= t1),
            }
        )
    return pd.DataFrame(rows)


def cascade_threshold_sweep(scored: pd.DataFrame, fold_id: str, variant_id: str) -> pd.DataFrame:
    day_df = scored.groupby(["substation_id", "date"], as_index=False).agg(
        label_day=("label_interval", "any"),
        prob_day=("prob_day", "first"),
        max_prob_interval=("prob_interval", "max"),
    )
    day_df["max_prob_interval"] = day_df["max_prob_interval"].fillna(-np.inf)
    rows = []
    for t1 in XGB1_THRESHOLD_GRID:
        for t2 in XGB2_THRESHOLD_GRID:
            pred_day = (day_df["prob_day"] >= t1) & (day_df["max_prob_interval"] >= t2)
            rows.append(
                {
                    "fold_id": fold_id,
                    "variant_id": variant_id,
                    "xgb1_threshold": float(t1),
                    "xgb2_threshold": float(t2),
                    **binary_metrics(day_df["label_day"], pred_day),
                }
            )
    return pd.DataFrame(rows)


def sort_threshold_candidates(sweep: pd.DataFrame) -> pd.DataFrame:
    return sweep.sort_values(
        ["f1", "recall", "precision", "fp"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)


def choose_best_global_thresholds(scored: pd.DataFrame, fold_id: str, variant_id: str) -> tuple[dict[str, Any], pd.DataFrame]:
    sweep = cascade_threshold_sweep(scored, fold_id, variant_id)
    return sort_threshold_candidates(sweep).iloc[0].to_dict(), sweep


def choose_site_thresholds(scored: pd.DataFrame, fold_id: str, variant_id: str, fallback: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for site, site_df in scored.groupby("substation_id"):
        day_df = site_df.groupby(["substation_id", "date"], as_index=False).agg(
            label_day=("label_interval", "any"),
            prob_day=("prob_day", "first"),
            max_prob_interval=("prob_interval", "max"),
        )
        positive_days = int(day_df["label_day"].sum())
        negative_days = int((~day_df["label_day"].astype(bool)).sum())
        if positive_days < SITE_MIN_POSITIVE_DAYS_FOR_SITE_THRESHOLD or negative_days < SITE_MIN_POSITIVE_DAYS_FOR_SITE_THRESHOLD:
            rows.append(
                {
                    "fold_id": fold_id,
                    "variant_id": variant_id,
                    "substation_id": site,
                    "xgb1_threshold": float(fallback["xgb1_threshold"]),
                    "xgb2_threshold": float(fallback["xgb2_threshold"]),
                    "source": "fallback_global_low_support",
                    "tune_positive_days": positive_days,
                    "tune_negative_days": negative_days,
                }
            )
            continue
        best, _ = choose_best_global_thresholds(site_df, f"{fold_id}_{site}", variant_id)
        rows.append(
            {
                "fold_id": fold_id,
                "variant_id": variant_id,
                "substation_id": site,
                "xgb1_threshold": float(best["xgb1_threshold"]),
                "xgb2_threshold": float(best["xgb2_threshold"]),
                "source": "site_tuned",
                "tune_positive_days": positive_days,
                "tune_negative_days": negative_days,
            }
        )
    return pd.DataFrame(rows)


def site_level_metrics(pred: pd.DataFrame, label: str, variant_id: str, calibration: str) -> pd.DataFrame:
    frames = []
    for site, site_df in pred.groupby("substation_id"):
        metrics = evaluate_predicted_frame(site_df, "Beta", label, label, variant_id, calibration)
        metrics["substation_id"] = site
        frames.append(metrics)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## Train And Score Variants

Each variant is trained on Alpha and scored on Beta. The baseline uses the
current config and raw features. Site-normalised variants scale net load and
solar before feature generation but evaluate predictions against the original
Beta labels.


In [ ]:
@dataclass
class VariantSpec:
    variant_id: str
    result_type: str
    description: str
    scale_mode: str | None = None
    hyper_updates: dict[str, Any] | None = None


variant_specs = [
    VariantSpec(
        variant_id="baseline_raw_features",
        result_type="baseline_current_config",
        description="Current m8_xgb hyperparameters with raw Alpha/Beta features.",
    )
]
if RUN_SITE_NORMALISED_VARIANTS:
    for item in SITE_NORMALISATION_VARIANTS:
        variant_specs.append(
            VariantSpec(
                variant_id=item["variant_id"],
                result_type="site_normalised_variant",
                description=item["description"],
                scale_mode=item["scale_mode"],
            )
        )
if RUN_LIGHT_HYPERPARAMETER_VARIANTS:
    for item in LIGHT_HYPERPARAMETER_VARIANTS:
        variant_specs.append(
            VariantSpec(
                variant_id=item["variant_id"],
                result_type="light_hyperparameter_variant",
                description=item["description"],
                hyper_updates=item["updates"],
            )
        )

display(pd.DataFrame([spec.__dict__ for spec in variant_specs]))


In [ ]:
scored_variants: dict[str, pd.DataFrame] = {}
variant_metadata = []

for spec in variant_specs:
    started = time.time()
    print(f"\n=== Training/scoring variant: {spec.variant_id} ===")
    run_cfg = make_cfg_with_hyperparams(cfg, spec.hyper_updates)
    train_alpha = transform_variant(alpha, spec.scale_mode)
    score_beta_features = transform_variant(beta, spec.scale_mode)
    bundle = h.train_m8_bundle(train_alpha, run_cfg, ARTICLE_ROOT)
    scored = score_m8_probabilities(score_beta_features, beta, bundle, run_cfg, score_xgb2_all_days=True)
    scored_variants[spec.variant_id] = scored
    elapsed = time.time() - started
    variant_metadata.append(
        {
            "variant_id": spec.variant_id,
            "result_type": spec.result_type,
            "description": spec.description,
            "scale_mode": spec.scale_mode or "none",
            "hyper_updates_json": json.dumps(spec.hyper_updates or {}, sort_keys=True),
            "elapsed_seconds": round(elapsed, 2),
            "rows_scored": int(len(scored)),
            "prob_day_non_null": int(scored["prob_day"].notna().sum()),
            "prob_interval_non_null": int(scored["prob_interval"].notna().sum()),
        }
    )
    print(f"Done in {elapsed:.1f}s | interval probabilities: {scored['prob_interval'].notna().sum():,}")

variant_metadata_df = pd.DataFrame(variant_metadata)
write_csv(variant_metadata_df, "00_variant_metadata.csv")
display(variant_metadata_df)


## Baseline Fidelity Check

This verifies that the notebook-local probability scorer reproduces Notebook 2
when the current config thresholds are applied.


In [ ]:
baseline_scored = scored_variants["baseline_raw_features"]
baseline_pred = apply_global_thresholds(baseline_scored, baseline_t1, baseline_t2)
baseline_metrics = evaluate_predicted_frame(
    baseline_pred,
    dataset="Beta",
    fold_id="all_beta_current_config",
    result_type="baseline_current_config",
    variant_id="baseline_raw_features",
    calibration="current_config",
)

notebook2_metrics_path = ARTICLE_ROOT / "outputs" / "metrics" / "02_correction_validation" / "01_correction_metrics.csv"
if notebook2_metrics_path.exists():
    nb2 = pd.read_csv(notebook2_metrics_path)
    nb2_beta_m8 = nb2[(nb2["dataset"] == "Beta") & (nb2["method"] == "m8_xgb")][
        ["level", "precision", "recall", "f1", "tp", "fp", "fn", "tn"]
    ].copy()
    check = baseline_metrics.merge(nb2_beta_m8, on="level", suffixes=("_diagnostic", "_notebook2"))
    for col in ["precision", "recall", "f1"]:
        check[f"{col}_abs_diff"] = (check[f"{col}_diagnostic"] - check[f"{col}_notebook2"]).abs()
else:
    check = pd.DataFrame({"warning": ["Notebook 2 metrics CSV not found; fidelity check skipped."]})

write_csv(baseline_metrics, "01_baseline_metrics.csv")
write_csv(check, "01b_baseline_fidelity_check.csv")
display(baseline_metrics)
display(check)


## Threshold Diagnostics And Split-Beta Search

For every variant, this section writes XGB1 threshold sweeps, final cascade
threshold sweeps, split-validated metrics, per-site threshold tables, and
optimistic all-Beta upper-bound metrics.


In [ ]:
xgb1_sweep_frames = []
cascade_sweep_frames = []
validation_metric_frames = []
site_threshold_frames = []
upper_bound_frames = []
selected_candidate_rows = []

for spec in variant_specs:
    scored_all = scored_variants[spec.variant_id]
    xgb1_sweep_frames.append(xgb1_threshold_sweep(scored_all, "all_beta", spec.variant_id))

    all_best, all_sweep = choose_best_global_thresholds(scored_all, "all_beta", spec.variant_id)
    cascade_sweep_frames.append(all_sweep.assign(scope="all_beta"))
    upper_pred = apply_global_thresholds(scored_all, all_best["xgb1_threshold"], all_best["xgb2_threshold"])
    upper_metrics = evaluate_predicted_frame(
        upper_pred,
        dataset="Beta",
        fold_id="all_beta_optimistic",
        result_type="all_beta_upper_bound",
        variant_id=spec.variant_id,
        calibration="global_thresholds_optimistic",
    )
    upper_metrics["xgb1_threshold"] = float(all_best["xgb1_threshold"])
    upper_metrics["xgb2_threshold"] = float(all_best["xgb2_threshold"])
    upper_bound_frames.append(upper_metrics)

    for fold in BETA_FOLDS:
        tune_df = scored_all.loc[date_mask(scored_all, fold["tune_start"], fold["tune_end"])].copy()
        eval_df = scored_all.loc[date_mask(scored_all, fold["eval_start"], fold["eval_end"])].copy()

        current_pred = apply_global_thresholds(eval_df, baseline_t1, baseline_t2)
        current_type = "baseline_current_config" if spec.variant_id == "baseline_raw_features" else spec.result_type
        current_metrics = evaluate_predicted_frame(
            current_pred,
            dataset="Beta",
            fold_id=fold["fold_id"],
            result_type=current_type,
            variant_id=spec.variant_id,
            calibration="current_config_thresholds",
        )
        current_metrics["xgb1_threshold"] = baseline_t1
        current_metrics["xgb2_threshold"] = baseline_t2
        validation_metric_frames.append(current_metrics)

        best_global, fold_sweep = choose_best_global_thresholds(tune_df, fold["fold_id"], spec.variant_id)
        cascade_sweep_frames.append(fold_sweep.assign(scope="split_tune"))
        global_pred = apply_global_thresholds(eval_df, best_global["xgb1_threshold"], best_global["xgb2_threshold"])
        global_type = "threshold_calibrated" if spec.variant_id == "baseline_raw_features" else spec.result_type
        global_metrics = evaluate_predicted_frame(
            global_pred,
            dataset="Beta",
            fold_id=fold["fold_id"],
            result_type=global_type,
            variant_id=spec.variant_id,
            calibration="global_thresholds_split_validated",
        )
        global_metrics["xgb1_threshold"] = float(best_global["xgb1_threshold"])
        global_metrics["xgb2_threshold"] = float(best_global["xgb2_threshold"])
        validation_metric_frames.append(global_metrics)
        selected_candidate_rows.append(
            {
                "fold_id": fold["fold_id"],
                "variant_id": spec.variant_id,
                "calibration": "global_thresholds_split_validated",
                "result_type": global_type,
                "xgb1_threshold": float(best_global["xgb1_threshold"]),
                "xgb2_threshold": float(best_global["xgb2_threshold"]),
                "tune_day_f1": float(best_global["f1"]),
                "tune_day_precision": float(best_global["precision"]),
                "tune_day_recall": float(best_global["recall"]),
            }
        )

        site_thresholds = choose_site_thresholds(tune_df, fold["fold_id"], spec.variant_id, best_global)
        site_threshold_frames.append(site_thresholds)
        site_pred = apply_site_thresholds(eval_df, site_thresholds)
        site_type = "site_threshold_calibrated" if spec.variant_id == "baseline_raw_features" else spec.result_type
        site_metrics = evaluate_predicted_frame(
            site_pred,
            dataset="Beta",
            fold_id=fold["fold_id"],
            result_type=site_type,
            variant_id=spec.variant_id,
            calibration="site_thresholds_split_validated",
        )
        site_metrics["xgb1_threshold"] = float(site_thresholds["xgb1_threshold"].median())
        site_metrics["xgb2_threshold"] = float(site_thresholds["xgb2_threshold"].median())
        validation_metric_frames.append(site_metrics)

xgb1_sweeps = pd.concat(xgb1_sweep_frames, ignore_index=True)
cascade_sweeps = pd.concat(cascade_sweep_frames, ignore_index=True)
validation_metrics = pd.concat(validation_metric_frames, ignore_index=True)
site_thresholds_all = pd.concat(site_threshold_frames, ignore_index=True)
upper_bound_metrics = pd.concat(upper_bound_frames, ignore_index=True)
selected_candidates = pd.DataFrame(selected_candidate_rows)

write_csv(xgb1_sweeps, "02_xgb1_threshold_sweep.csv")
write_csv(cascade_sweeps, "03_cascade_threshold_sweep.csv")
write_csv(validation_metrics, "04_fold_validation_metrics.csv")
write_csv(site_thresholds_all, "05_site_thresholds.csv")
write_csv(upper_bound_metrics, "06_all_beta_upper_bound_metrics.csv")
write_csv(selected_candidates, "07_selected_global_threshold_candidates.csv")

validation_day = validation_metrics[validation_metrics["level"] == "day"].copy()
leaderboard = (
    validation_day.groupby(["result_type", "variant_id", "calibration"], as_index=False)
    .agg(
        mean_day_f1=("f1", "mean"),
        mean_day_precision=("precision", "mean"),
        mean_day_recall=("recall", "mean"),
        total_tp=("tp", "sum"),
        total_fp=("fp", "sum"),
        total_fn=("fn", "sum"),
        folds=("fold_id", "nunique"),
    )
    .sort_values(["mean_day_f1", "mean_day_recall", "total_fp"], ascending=[False, False, True])
    .reset_index(drop=True)
)
write_csv(leaderboard, "08_ranked_leaderboard.csv")
display(leaderboard.head(20))


## Site-Level Before/After And Error Examples

This compares the current baseline with the highest-ranked split-validated
candidate and writes deterministic TP/TN/FP/FN examples for visual inspection.


In [ ]:
def candidate_pred_for_fold(row: pd.Series, fold: dict[str, str], scored: pd.DataFrame) -> pd.DataFrame:
    eval_df = scored.loc[date_mask(scored, fold["eval_start"], fold["eval_end"])].copy()
    if row["calibration"] == "site_thresholds_split_validated":
        thresholds = site_thresholds_all[
            (site_thresholds_all["fold_id"] == row["fold_id"]) &
            (site_thresholds_all["variant_id"] == row["variant_id"])
        ]
        return apply_site_thresholds(eval_df, thresholds)
    return apply_global_thresholds(eval_df, float(row["xgb1_threshold"]), float(row["xgb2_threshold"]))


best_key = leaderboard.iloc[0]
print("Best split-validated candidate:")
display(best_key.to_frame().T)

site_metric_frames = []
error_example_frames = []
best_pred_frames = []

for fold in BETA_FOLDS:
    eval_base = baseline_scored.loc[date_mask(baseline_scored, fold["eval_start"], fold["eval_end"])].copy()
    base_pred = apply_global_thresholds(eval_base, baseline_t1, baseline_t2)
    site_metric_frames.append(site_level_metrics(base_pred, "baseline_current_config", "baseline_raw_features", "current_config_thresholds"))

    best_rows = validation_metrics[
        (validation_metrics["fold_id"] == fold["fold_id"]) &
        (validation_metrics["level"] == "day") &
        (validation_metrics["variant_id"] == best_key["variant_id"]) &
        (validation_metrics["calibration"] == best_key["calibration"])
    ]
    if best_rows.empty:
        continue
    best_row = best_rows.iloc[0]
    best_scored = scored_variants[str(best_key["variant_id"])]
    best_pred = candidate_pred_for_fold(best_row, fold, best_scored)
    best_pred_frames.append(best_pred)
    site_metric_frames.append(site_level_metrics(best_pred, "best_split_validated", str(best_key["variant_id"]), str(best_key["calibration"])))

    day_examples = best_pred.groupby(["substation_id", "date"], as_index=False).agg(
        label_day=("label_interval", "any"),
        pred_day=("pred_interval", "any"),
        label_intervals=("label_interval", "sum"),
        pred_intervals=("pred_interval", "sum"),
        max_prob_day=("prob_day", "max"),
        max_prob_interval=("prob_interval", "max"),
        solar_peak=("solar_MW", "max"),
        net_min=("net_load_MW", "min"),
    )
    day_examples["confusion"] = np.select(
        [
            day_examples["label_day"] & day_examples["pred_day"],
            (~day_examples["label_day"]) & (~day_examples["pred_day"]),
            (~day_examples["label_day"]) & day_examples["pred_day"],
            day_examples["label_day"] & (~day_examples["pred_day"]),
        ],
        ["TP", "TN", "FP", "FN"],
        default="",
    )
    selected = (
        day_examples.assign(score=lambda x: x["solar_peak"].fillna(0) + x["label_intervals"].fillna(0) + x["pred_intervals"].fillna(0))
        .sort_values(["substation_id", "confusion", "score"], ascending=[True, True, False])
        .groupby(["substation_id", "confusion"], as_index=False)
        .head(3)
    )
    selected["fold_id"] = fold["fold_id"]
    selected["candidate"] = "best_split_validated"
    error_example_frames.append(selected)

site_before_after = pd.concat(site_metric_frames, ignore_index=True) if site_metric_frames else pd.DataFrame()
error_examples = pd.concat(error_example_frames, ignore_index=True) if error_example_frames else pd.DataFrame()
best_pred_all = pd.concat(best_pred_frames, ignore_index=True) if best_pred_frames else pd.DataFrame()

write_csv(site_before_after, "09_site_level_before_after_metrics.csv")
write_csv(error_examples, "10_error_examples.csv")
display(site_before_after[site_before_after["level"] == "day"].head(30))
display(error_examples.head(30))


## Figures And Optional HTML Examples


In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


day_probs = baseline_scored.groupby(["substation_id", "date"], as_index=False).agg(
    label_day=("label_interval", "any"),
    prob_day=("prob_day", "first"),
)

fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharex=True, sharey=True)
for ax, (site, group) in zip(axes.ravel(), day_probs.groupby("substation_id")):
    ax.hist(group.loc[~group["label_day"], "prob_day"], bins=np.linspace(0, 1, 21), alpha=0.65, color=JCOL["light_grey"], label="No RPF")
    ax.hist(group.loc[group["label_day"], "prob_day"], bins=np.linspace(0, 1, 21), alpha=0.65, color=JCOL["orange"], label="RPF")
    ax.axvline(baseline_t1, color=JCOL["dark_blue"], linestyle="--", linewidth=1.2)
    ax.set_title(site, fontsize=11)
    ax.set_axisbelow(True)
    ax.grid(color=JCOL["light_white"], linewidth=0.6)
axes[0, 0].legend(fontsize=9)
fig.supxlabel("XGB1 day probability")
fig.supylabel("Site-day count")
fig.suptitle("Baseline XGB1 probability distribution by Beta site", fontsize=14)
savefig(FIG_DIR / "fig01_xgb1_probability_distributions_by_site.png")

fig, ax = plt.subplots(figsize=(8, 6))
curve_specs = [
    ("All Beta", day_probs, JCOL["dark_blue"]),
    ("beta_B", day_probs[day_probs["substation_id"] == "beta_B"], JCOL["orange"]),
    ("beta_G", day_probs[day_probs["substation_id"] == "beta_G"], JCOL["grey"]),
    ("beta_F", day_probs[day_probs["substation_id"] == "beta_F"], JCOL["light_grey"]),
    ("beta_D", day_probs[day_probs["substation_id"] == "beta_D"], "#8f5f2a"),
]
for label, frame, color in curve_specs:
    if frame["label_day"].nunique() < 2:
        continue
    precision, recall, _ = precision_recall_curve(frame["label_day"].astype(int), frame["prob_day"])
    ap = average_precision_score(frame["label_day"].astype(int), frame["prob_day"])
    ax.plot(recall, precision, label=f"{label} AP={ap:.2f}", linewidth=2, color=color)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.03)
ax.set_title("XGB1 candidate-day precision-recall curves")
ax.set_axisbelow(True)
ax.grid(color=JCOL["light_white"], linewidth=0.7)
ax.legend(fontsize=9)
savefig(FIG_DIR / "fig02_xgb1_precision_recall_curves.png")

heat = cascade_sweeps[
    (cascade_sweeps["variant_id"] == "baseline_raw_features") &
    (cascade_sweeps["fold_id"] == "all_beta") &
    (cascade_sweeps["scope"] == "all_beta")
].pivot(index="xgb2_threshold", columns="xgb1_threshold", values="f1")
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heat.to_numpy(), origin="lower", aspect="auto", cmap=h.journal_colormap("rpf_heat"))
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels([f"{v:.2f}" for v in heat.columns], rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([f"{v:.2f}" for v in heat.index])
ax.set_xlabel("XGB1 threshold")
ax.set_ylabel("XGB2 threshold")
ax.set_title("All-Beta optimistic final day F1 threshold surface")
fig.colorbar(im, ax=ax, label="Day F1")
savefig(FIG_DIR / "fig03_cascade_threshold_heatmap_baseline.png")

plot_leader = leaderboard.head(12).iloc[::-1].copy()
labels = plot_leader["variant_id"] + "\n" + plot_leader["calibration"].str.replace("_", " ")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(plot_leader)), plot_leader["mean_day_f1"], color=JCOL["dark_blue"])
ax.set_yticks(range(len(plot_leader)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlim(0, max(0.05, min(1.0, plot_leader["mean_day_f1"].max() * 1.15)))
ax.set_xlabel("Mean held-out Beta day F1")
ax.set_title("Split-validated m8_xgb candidate leaderboard")
ax.set_axisbelow(True)
ax.grid(axis="x", color=JCOL["light_white"], linewidth=0.7)
savefig(FIG_DIR / "fig04_split_validated_day_f1_leaderboard.png")

site_plot = site_before_after[site_before_after["level"] == "day"].copy()
if not site_plot.empty:
    site_plot["display"] = np.where(site_plot["calibration"] == "current_config_thresholds", "Current config", "Best candidate")
    pivot = (
        site_plot.groupby(["substation_id", "display"], as_index=False)["f1"]
        .mean()
        .pivot(index="substation_id", columns="display", values="f1")
        .fillna(0)
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(pivot.index))
    width = 0.36
    ax.bar(x - width / 2, pivot.get("Current config", pd.Series(0, index=pivot.index)), width, label="Current config", color=JCOL["light_grey"])
    ax.bar(x + width / 2, pivot.get("Best candidate", pd.Series(0, index=pivot.index)), width, label="Best candidate", color=JCOL["orange"])
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Day F1")
    ax.set_title("Beta site-level day F1 before and after best split-validated candidate")
    ax.legend()
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=JCOL["light_white"], linewidth=0.7)
    savefig(FIG_DIR / "fig05_beta_site_before_after_day_f1.png")

def pooled_cm(rows: pd.DataFrame) -> np.ndarray:
    return np.array([[int(rows["tn"].sum()), int(rows["fp"].sum())], [int(rows["fn"].sum()), int(rows["tp"].sum())]])

current_rows = validation_metrics[
    (validation_metrics["level"] == "day") &
    (validation_metrics["variant_id"] == "baseline_raw_features") &
    (validation_metrics["calibration"] == "current_config_thresholds")
]
best_rows = validation_metrics[
    (validation_metrics["level"] == "day") &
    (validation_metrics["variant_id"] == best_key["variant_id"]) &
    (validation_metrics["calibration"] == best_key["calibration"])
]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, title, rows in [(axes[0], "Current config", current_rows), (axes[1], "Best split-validated", best_rows)]:
    cm = pooled_cm(rows)
    ax.imshow(cm, cmap=h.journal_colormap("correction_confusion"))
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, f"{value:,}", ha="center", va="center", color="white" if value > cm.max() / 2 else JCOL["dark_blue"], fontsize=12, fontweight="bold")
    ax.set_xticks([0, 1], labels=["Pred no", "Pred RPF"])
    ax.set_yticks([0, 1], labels=["Actual no", "Actual RPF"])
    ax.set_title(title)
fig.suptitle("Held-out Beta day-level confusion matrices")
savefig(FIG_DIR / "fig06_day_confusion_current_vs_best.png")

print(f"Figures written to {FIG_DIR}")


In [ ]:
html_index_rows = []
if WRITE_PLOTLY_HTML_EXAMPLES and not best_pred_all.empty:
    try:
        import plotly.graph_objects as go

        focus_sites = ["beta_B", "beta_G", "beta_F", "beta_D"]
        day_conf = best_pred_all.groupby(["substation_id", "date"], as_index=False).agg(
            label_day=("label_interval", "any"),
            pred_day=("pred_interval", "any"),
            label_intervals=("label_interval", "sum"),
            pred_intervals=("pred_interval", "sum"),
            solar_peak=("solar_MW", "max"),
        )
        day_conf["confusion"] = np.select(
            [
                day_conf["label_day"] & day_conf["pred_day"],
                (~day_conf["label_day"]) & (~day_conf["pred_day"]),
                (~day_conf["label_day"]) & day_conf["pred_day"],
                day_conf["label_day"] & (~day_conf["pred_day"]),
            ],
            ["TP", "TN", "FP", "FN"],
            default="",
        )
        for site in focus_sites:
            for group in ["TP", "TN", "FP", "FN"]:
                pick = day_conf[(day_conf["substation_id"] == site) & (day_conf["confusion"] == group)].copy()
                if pick.empty:
                    continue
                pick = pick.assign(score=pick["solar_peak"] + pick["label_intervals"] + pick["pred_intervals"]).sort_values("score", ascending=False).head(2)
                for _, row in pick.iterrows():
                    one = best_pred_all[(best_pred_all["substation_id"] == site) & (best_pred_all["date"] == row["date"])].copy()
                    one["timestamp_plot"] = h._parse_wall_clock(one["timestamp"])
                    fig = go.Figure()
                    fig.add_trace(go.Scatter(x=one["timestamp_plot"], y=one["net_load_MW"], name="Raw net load", line=dict(color=JCOL["dark_blue"])))
                    fig.add_trace(go.Scatter(x=one["timestamp_plot"], y=one["solar_MW"], name="Solar", line=dict(color=JCOL["orange"]), yaxis="y2"))
                    true_points = one[one["label_interval"]]
                    pred_points = one[one["pred_interval"]]
                    fig.add_trace(go.Scatter(x=true_points["timestamp_plot"], y=true_points["net_load_MW"], name="True RPF intervals", mode="markers", marker=dict(color=JCOL["grey"], size=8)))
                    fig.add_trace(go.Scatter(x=pred_points["timestamp_plot"], y=pred_points["net_load_MW"], name="Predicted RPF intervals", mode="markers", marker=dict(color="#d9541e", size=9, symbol="x")))
                    fig.update_layout(
                        title=f"{site} {row['date']} | {group} | {best_key['variant_id']}",
                        xaxis_title="Time",
                        yaxis_title="Net load (MW)",
                        yaxis2=dict(title="Solar (MW)", overlaying="y", side="right", showgrid=False),
                        template="plotly_white",
                        legend=dict(orientation="h"),
                    )
                    filename = f"{site}_{row['date']}_{group}_{best_key['variant_id']}.html".replace(":", "")
                    path = HTML_DIR / filename
                    fig.write_html(path, include_plotlyjs="cdn")
                    html_index_rows.append({"substation_id": site, "date": row["date"], "confusion": group, "variant_id": best_key["variant_id"], "path": str(path)})
    except Exception as exc:
        html_index_rows.append({"warning": f"Plotly HTML generation skipped: {exc}"})

html_index = pd.DataFrame(html_index_rows)
write_csv(html_index, "11_plotly_html_index.csv")
display(html_index.head(20))


## Manifest And Leaderboard


In [ ]:
manifest = {
    "notebook": "02_beta_m8_xgb_threshold_and_variant_search.ipynb",
    "created_at_local": pd.Timestamp.now().isoformat(),
    "article_root": str(ARTICLE_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "baseline_thresholds": {"xgb1": baseline_t1, "xgb2": baseline_t2},
    "beta_folds": BETA_FOLDS,
    "threshold_grid": {
        "xgb1": [float(v) for v in XGB1_THRESHOLD_GRID],
        "xgb2": [float(v) for v in XGB2_THRESHOLD_GRID],
    },
    "variant_count": len(variant_specs),
    "variants": [spec.__dict__ for spec in variant_specs],
    "best_split_validated_candidate": leaderboard.iloc[0].to_dict(),
    "warning": "Exploratory misc notebook. Do not treat all_beta_upper_bound rows as publication-ready validation.",
}
manifest_path = OUTPUT_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Output files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(OUTPUT_ROOT))

print("\nBest split-validated candidates:")
display(leaderboard.head(10))
